In [ ]:
import os

REPO = 'giomamaca/walmart-sales-forecasting'

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    import shutil
    from google.colab import userdata, files

    kaggle_dir = os.path.expanduser('~/.kaggle')
    os.makedirs(kaggle_dir, exist_ok=True)
    if not os.path.exists(f'{kaggle_dir}/kaggle.json'):
        print('Upload your kaggle.json')
        uploaded = files.upload()
        shutil.move('kaggle.json', f'{kaggle_dir}/kaggle.json')
        os.chmod(f'{kaggle_dir}/kaggle.json', 0o600)

    github_token = userdata.get('GITHUB_TOKEN')
    repo_dir = f'/content/{REPO.split("/")[1]}'
    if not os.path.exists(repo_dir):
        !git clone -q https://{github_token}@github.com/{REPO}.git {repo_dir}
    os.chdir(repo_dir)

    !pip install -q kaggle mlflow wandb lightgbm

    import wandb
    wandb.login(key=userdata.get('WANDB_API_KEY'))

    !kaggle competitions download -c walmart-recruiting-store-sales-forecasting -p data
    !unzip -o -q data/walmart-recruiting-store-sales-forecasting.zip -d data
    !for f in data/*.zip; do unzip -o -q "$f" -d data; done

    !mlflow db upgrade sqlite:///mlflow.db

    DATA_DIR = 'data'
else:
    DATA_DIR = '..'

In [ ]:
import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn
import wandb
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from lightgbm import LGBMRegressor

FIXED = dict(verbose=-1)

In [ ]:
train = pd.read_csv(f'{DATA_DIR}/train.csv', parse_dates=['Date'])
features = pd.read_csv(f'{DATA_DIR}/features.csv', parse_dates=['Date'])
stores = pd.read_csv(f'{DATA_DIR}/stores.csv')

train.shape, features.shape, stores.shape

In [ ]:
def wmae(y_true, y_pred, is_holiday):
    weights = np.where(is_holiday, 5, 1)
    return np.average(np.abs(y_true - y_pred), weights=weights)

In [ ]:
MARKDOWN_COLS = ['MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5']
PRE_CHRISTMAS = pd.to_datetime(['2010-12-24', '2011-12-23', '2012-12-21'])

FEATURE_COLS = [
    'Store', 'Dept', 'IsHoliday', 'Size', 'Type',
    'Temperature', 'Fuel_Price', 'CPI', 'Unemployment',
    'MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5',
    'Year', 'Month', 'Week', 'IsPreChristmas',
    'Sales_Lag52', 'Sales_Lag52_Mean3',
    'Series_Mean', 'Series_Median', 'Series_Std', 'Series_WoyMean',
    'Dept_Mean', 'Store_Mean',
]

# v2: v1-is featurebs emateba istoriuli feature-ebi. yvela lag >= 51 kviraa,
# amitom test-is nebismieri tarigistvis mnishvneloba train-is istoriidan modis
# da rekursiuli prognozi ar gvchirdeba.
class FeatureBuilderV2(BaseEstimator, TransformerMixin):
    def __init__(self, stores_df, features_df):
        self.stores_df = stores_df
        self.features_df = features_df

    def fit(self, X, y=None):
        df = X[['Store', 'Dept', 'Date']].copy()
        df['Date'] = pd.to_datetime(df['Date'])
        df['y'] = y.values if hasattr(y, 'values') else y

        self.history_ = df.set_index(['Store', 'Dept', 'Date'])['y']
        self.series_stats_ = df.groupby(['Store', 'Dept'])['y'].agg(['mean', 'median', 'std'])
        woy = df['Date'].dt.isocalendar().week.astype(int)
        self.woy_mean_ = df.assign(woy=woy).groupby(['Store', 'Dept', 'woy'])['y'].mean()
        self.dept_mean_ = df.groupby('Dept')['y'].mean()
        self.store_mean_ = df.groupby('Store')['y'].mean()
        return self

    def _lag(self, df, weeks):
        idx = pd.MultiIndex.from_arrays(
            [df['Store'], df['Dept'], df['Date'] - pd.Timedelta(weeks=weeks)])
        return self.history_.reindex(idx).values

    def transform(self, X):
        df = X.copy()
        df['Date'] = pd.to_datetime(df['Date'])
        df = df.merge(self.stores_df, on='Store', how='left')
        df = df.merge(self.features_df.drop(columns=['IsHoliday']), on=['Store', 'Date'], how='left')

        df[MARKDOWN_COLS] = df[MARKDOWN_COLS].fillna(0)
        df['Type'] = df['Type'].map({'A': 0, 'B': 1, 'C': 2})
        df['IsHoliday'] = df['IsHoliday'].astype(int)

        df['Year'] = df['Date'].dt.year
        df['Month'] = df['Date'].dt.month
        df['Week'] = df['Date'].dt.isocalendar().week.astype(int)
        df['IsPreChristmas'] = df['Date'].isin(PRE_CHRISTMAS).astype(int)

        df['Sales_Lag52'] = self._lag(df, 52)
        lags = pd.DataFrame({w: self._lag(df, w) for w in (51, 52, 53)})
        df['Sales_Lag52_Mean3'] = lags.mean(axis=1).values

        pair_idx = pd.MultiIndex.from_arrays([df['Store'], df['Dept']])
        stats = self.series_stats_.reindex(pair_idx)
        df['Series_Mean'] = stats['mean'].values
        df['Series_Median'] = stats['median'].values
        df['Series_Std'] = stats['std'].values

        woy_idx = pd.MultiIndex.from_arrays(
            [df['Store'], df['Dept'], df['Date'].dt.isocalendar().week.astype(int)])
        df['Series_WoyMean'] = self.woy_mean_.reindex(woy_idx).values
        df['Dept_Mean'] = self.dept_mean_.reindex(df['Dept']).values
        df['Store_Mean'] = self.store_mean_.reindex(df['Store']).values

        return df[FEATURE_COLS]

In [ ]:
mlflow.set_tracking_uri('sqlite:///mlflow.db')
mlflow.set_experiment('LightGBM_v3_Training')

In [ ]:
with mlflow.start_run(run_name='LightGBM_v3_Cleaning'):
    wandb.init(project='walmart-sales-forecasting', name='LightGBM_v3_Cleaning', reinit='finish_previous')

    n_negative = int((train['Weekly_Sales'] < 0).sum())
    missing_markdown_pct = float(features[MARKDOWN_COLS].isna().mean().mean())

    mlflow.log_param('negative_sales_rows', n_negative)
    mlflow.log_param('negative_sales_action', 'clip_to_zero')
    mlflow.log_metric('missing_markdown_pct', missing_markdown_pct)
    wandb.log({'negative_sales_rows': n_negative, 'missing_markdown_pct': missing_markdown_pct})
    wandb.finish()

    y_train = train['Weekly_Sales'].clip(lower=0)

In [ ]:
def time_based_splits(dates, n_splits=3):
    dates = np.sort(dates.unique())
    fold_size = len(dates) // (n_splits + 1)
    splits = []
    for i in range(1, n_splits + 1):
        train_end = dates[fold_size * i - 1]
        val_end = dates[min(fold_size * (i + 1) - 1, len(dates) - 1)]
        splits.append((train_end, val_end))
    return splits

splits = time_based_splits(train['Date'])
splits

In [ ]:
SEARCH_SPACE = {
    'n_estimators': [500, 1000, 1500],
    'num_leaves': [31, 63, 127, 255],
    'learning_rate': [0.03, 0.05, 0.1],
    'min_child_samples': [20, 50, 100, 200],
    'subsample': [0.7, 0.8, 1.0],
    'colsample_bytree': [0.7, 0.8, 1.0],
}
N_TRIALS = 15

rng = np.random.default_rng(42)

def sample_params(rng):
    return {k: v[rng.integers(len(v))] for k, v in SEARCH_SPACE.items()}

fold_data = []
for train_end, val_end in splits:
    tm = train['Date'] <= train_end
    vm = (train['Date'] > train_end) & (train['Date'] <= val_end)
    b = FeatureBuilderV2(stores, features)
    b.fit(train.loc[tm, ['Store', 'Dept', 'Date']], y_train[tm])
    fold_data.append((
        b.transform(train[tm]), y_train[tm],
        b.transform(train[vm]), y_train[vm],
        train.loc[vm, 'IsHoliday'].values,
    ))

def cv_score(params):
    scores = []
    for X_tr, y_tr, X_val, y_val, hol in fold_data:
        model = LGBMRegressor(**params, **FIXED)
        model.fit(X_tr, y_tr)
        preds = model.predict(X_val)
        scores.append(wmae(y_val.values, preds, hol))
    return scores

with mlflow.start_run(run_name='LightGBM_v3_Tuning'):
    wandb.init(project='walmart-sales-forecasting', name='LightGBM_v3_Tuning', reinit='finish_previous')

    best_wmae, best_params = None, None
    for t in range(N_TRIALS):
        params = sample_params(rng)
        scores = cv_score(params)
        mean = float(np.mean(scores))
        mlflow.log_metric('trial_wmae_mean', mean, step=t)
        wandb.log({'trial': t, 'trial_wmae_mean': mean, **params})
        if best_wmae is None or mean < best_wmae:
            best_wmae, best_params = mean, params
        print(t, round(mean, 1), params)

    mlflow.log_metric('best_wmae_mean', best_wmae)
    mlflow.log_dict(best_params, 'best_params.json')
    wandb.log({'best_wmae_mean': best_wmae})
    wandb.finish()

best_wmae, best_params

In [ ]:
with mlflow.start_run(run_name='LightGBM_v3_CV'):
    mlflow.log_params(best_params)
    wandb.init(project='walmart-sales-forecasting', name='LightGBM_v3_CV', config=best_params, reinit='finish_previous')

    fold_scores = cv_score(best_params)
    for i, s in enumerate(fold_scores):
        mlflow.log_metric('wmae', s, step=i)
        wandb.log({'fold': i, 'wmae': s})

    mlflow.log_metric('wmae_mean', float(np.mean(fold_scores)))
    mlflow.log_metric('wmae_std', float(np.std(fold_scores)))
    wandb.log({'wmae_mean': float(np.mean(fold_scores)), 'wmae_std': float(np.std(fold_scores))})
    wandb.finish()

fold_scores

In [ ]:
with mlflow.start_run(run_name='LightGBM_v3_Final'):
    mlflow.log_params(best_params)
    wandb.init(project='walmart-sales-forecasting', name='LightGBM_v3_Final', config=best_params, reinit='finish_previous')

    pipeline = Pipeline([
        ('features', FeatureBuilderV2(stores, features)),
        ('model', LGBMRegressor(**best_params, **FIXED)),
    ])
    pipeline.fit(train[['Store', 'Dept', 'Date', 'IsHoliday']], y_train)

    preds = pipeline.predict(train[['Store', 'Dept', 'Date', 'IsHoliday']])
    train_wmae = wmae(y_train.values, preds, train['IsHoliday'].values)
    mlflow.log_metric('train_wmae', train_wmae)
    wandb.log({'train_wmae': train_wmae})

    mlflow.sklearn.log_model(pipeline, name='model', serialization_format='cloudpickle')
    wandb.finish()

train_wmae

In [ ]:
if IN_COLAB:
    import requests
    gh_user = requests.get('https://api.github.com/user', headers={'Authorization': f'token {github_token}'}).json()['login']
    !git config user.name "{gh_user}"
    !git config user.email "{gh_user}@users.noreply.github.com"

    !git add mlflow.db mlruns
    !git commit -m "Run model_experiment_LightGBM_v3.ipynb in Colab"
    !git push